<a href="https://colab.research.google.com/github/Bernpro/Property-Hunter-Jarvis-chat-version/blob/main/Jarvis_para_Propiedades_version_chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ============================================
# JARVIS CON BÚSQUEDA DE PROPIEDADES EN USA
# VERSIÓN QUE FUNCIONA EN GOOGLE COLAB
# ============================================

# Celda 1: Instalar librerías necesarias
!pip install -q requests

import datetime
import random
import urllib.parse
import requests
import json
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# Limpiar
clear_output(wait=True)

print("="*55)
print("🤖 JARVIS - ASISTENTE CON BÚSQUEDA DE PROPIEDADES EN USA")
print("="*55)
print("✅ Busca casas en venta en cualquier ciudad de Estados Unidos")
print("✅ Totalmente GRATIS - Sin necesidad de API key")
print("="*55)

# ============================================
# FUNCIÓN PARA BUSCAR PROPIEDADES (SIN API KEY)
# ============================================

def buscar_propiedades(ubicacion, limit=5):
    """
    Busca propiedades en Zillow usando enlaces directos
    (Funciona siempre, sin necesidad de API)
    """
    ubicacion_codificada = urllib.parse.quote(ubicacion)

    # Crear URLs de búsqueda en Zillow
    urls = {
        'general': f"https://www.zillow.com/{ubicacion_codificada}/",
        'venta': f"https://www.zillow.com/{ubicacion_codificada}/for_sale/",
        'mapa': f"https://www.zillow.com/homes/{ubicacion_codificada}_rb/"
    }

    # Generar propiedades de ejemplo (simuladas para mostrar formato)
    # En un entorno real, Zillow requiere JavaScript, así que mostramos enlaces
    propiedades = []

    # Mapeo de ciudades populares
    ciudades_info = {
        'miami': {'precio_medio': '450,000', 'rango': '300k - 800k'},
        'orlando': {'precio_medio': '380,000', 'rango': '250k - 600k'},
        'los angeles': {'precio_medio': '950,000', 'rango': '600k - 1.5M'},
        'nueva york': {'precio_medio': '750,000', 'rango': '500k - 1.2M'},
        'texas': {'precio_medio': '350,000', 'rango': '250k - 550k'},
        'california': {'precio_medio': '800,000', 'rango': '500k - 1.3M'},
        'florida': {'precio_medio': '400,000', 'rango': '280k - 650k'},
        'chicago': {'precio_medio': '320,000', 'rango': '200k - 500k'},
        'houston': {'precio_medio': '310,000', 'rango': '220k - 480k'},
        'phoenix': {'precio_medio': '420,000', 'rango': '300k - 600k'}
    }

    ubicacion_lower = ubicacion.lower()

    if ubicacion_lower in ciudades_info:
        info = ciudades_info[ubicacion_lower]
        precio_medio = info['precio_medio']
        rango = info['rango']

        for i in range(min(limit, 5)):
            precio = f"${int(precio_medio.replace(',', '')) + (i * 25000):,}"
            propiedades.append({
                'direccion': f"{i+1}. Propiedad de muestra en {ubicacion.title()}",
                'precio': precio,
                'habitaciones': random.choice([2, 3, 4, 5]),
                'baños': random.choice([1, 2, 3, 4]),
                'superficie': f"{random.choice([1200, 1500, 1800, 2200, 2800])} sqft"
            })
    else:
        # Para ubicaciones no predefinidas
        for i in range(limit):
            propiedades.append({
                'direccion': f"{i+1}. {ubicacion.title()} - Opción de búsqueda {i+1}",
                'precio': f"${random.randint(250000, 800000):,}",
                'habitaciones': random.choice([2, 3, 4, 5]),
                'baños': random.choice([1, 2, 3, 4]),
                'superficie': f"{random.choice([1200, 1500, 1800, 2200, 2800])} sqft"
            })

    return {
        'exito': True,
        'ubicacion': ubicacion,
        'propiedades': propiedades,
        'urls': urls
    }


def buscar_propiedades_zillow_directo(ubicacion):
    """
    Genera enlaces directos a Zillow para búsqueda real
    """
    ubicacion_codificada = urllib.parse.quote(ubicacion)

    return {
        'url_principal': f"https://www.zillow.com/{ubicacion_codificada}/",
        'url_venta': f"https://www.zillow.com/{ubicacion_codificada}/for_sale/",
        'url_mapa': f"https://www.zillow.com/homes/{ubicacion_codificada}_rb/"
    }


# ============================================
# FUNCIÓN PRINCIPAL DE JARVIS
# ============================================

# Almacenar notas
notas_guardadas = []

def procesar_comando(comando):
    """Procesa comandos del usuario"""
    comando = comando.lower().strip()

    if not comando:
        return "❌ No escribiste nada"

    # ===== HORA =====
    if 'hora' in comando:
        ahora = datetime.datetime.now().strftime("%I:%M %p")
        return f"🕐 Son las {ahora}"

    # ===== FECHA =====
    elif 'fecha' in comando:
        hoy = datetime.datetime.now().strftime("%d/%m/%Y")
        return f"📅 Hoy es {hoy}"

    # ===== BUSCAR PROPIEDADES EN USA =====
    elif any(palabra in comando for palabra in ['propiedad', 'propiedades', 'casa', 'casas', 'real estate', 'zillow']):
        # Extraer ubicación
        ubicacion = comando
        for palabra in ['buscar', 'propiedad', 'propiedades', 'casa', 'casas', 'en', 'venta', 'real estate', 'zillow', 'busca']:
            ubicacion = ubicacion.replace(palabra, '')
        ubicacion = ubicacion.strip()

        if not ubicacion:
            return "❓ ¿En qué ciudad o estado quieres buscar propiedades?\n\nEjemplos:\n• buscar casas en Miami\n• propiedades en Texas\n• real estate Orlando"

        # Buscar propiedades
        resultado = buscar_propiedades(ubicacion, limit=5)
        urls = buscar_propiedades_zillow_directo(ubicacion)

        if resultado['exito']:
            respuesta = formatear_propiedades(resultado, urls)
            return respuesta
        else:
            return f"❌ No encontré propiedades en {ubicacion}. Intenta con otra ubicación como Miami, Orlando, Texas, etc."

    # ===== BUSCAR EN GOOGLE =====
    elif 'buscar' in comando and 'google' in comando:
        query = comando.replace('buscar', '').replace('google', '').strip()
        if query:
            url = f"https://www.google.com/search?q={urllib.parse.quote(query)}"
            display(HTML(f'<a href="{url}" target="_blank">🔍 Haz clic aquí para buscar: {query}</a>'))
            return f"🔍 Buscando: {query} - Haz clic en el enlace de arriba"
        else:
            return "❓ ¿Qué quieres buscar? Ejemplo: 'buscar google inteligencia artificial'"

    # ===== NOTAS =====
    elif 'nota' in comando and 'agregar' in comando:
        nota = comando.replace('agregar', '').replace('nota', '').strip()
        if nota:
            notas_guardadas.append(nota)
            return f"📝 Nota guardada: {nota}"
        else:
            return "❓ Ejemplo: 'agregar nota comprar leche'"

    elif 'mostrar notas' in comando:
        if notas_guardadas:
            notas_texto = "\n".join([f"  {i+1}. {n}" for i, n in enumerate(notas_guardadas)])
            return f"📋 Tus notas:\n{notas_texto}"
        else:
            return "📭 No hay notas guardadas"

    elif 'borrar notas' in comando:
        notas_guardadas.clear()
        return "🗑️ Todas las notas han sido borradas"

    # ===== CHISTE =====
    elif 'chiste' in comando:
        chistes = [
            "¿Qué le dice un techo a otro? Te echo de menos",
            "¿Cómo se llama el campeón de buceo japonés? Tokofondo",
            "¿Qué hace una abeja en el gimnasio? ¡Zum-ba!",
            "¿Por qué el libro de matemáticas era triste? Porque tenía muchos problemas",
            "¿Qué le dice un jardinero a otro? Nos vemos cuando podamos"
        ]
        return f"😂 {random.choice(chistes)}"

    # ===== CALCULADORA =====
    elif 'calcula' in comando or 'cuanto es' in comando:
        try:
            expresion = comando.replace('calcula', '').replace('cuanto es', '').replace('cuánto es', '')
            expresion = expresion.replace('más', '+').replace('menos', '-')
            expresion = expresion.replace('por', '*').replace('entre', '/')
            expresion = expresion.replace('x', '*')

            # Solo números y operadores básicos
            if all(c in '0123456789+-*/(). ' for c in expresion):
                resultado = eval(expresion)
                return f"🧮 {expresion} = {resultado}"
            else:
                return "❌ Solo operaciones básicas (+, -, *, /)"
        except:
            return "❌ No pude calcular. Ejemplo: 'calcula 5 + 3'"

    # ===== SALUDO =====
    elif comando in ['hola', 'buenos dias', 'buenas tardes']:
        saludos = ["¡Hola! ¿En qué puedo ayudarte?", "¡Hola! ¿Cómo estás?", "¡Saludos! Dime qué necesitas"]
        return f"🤖 {random.choice(saludos)}"

    # ===== LISTA DE ESTADOS =====
    elif 'estados' in comando and 'buscar' in comando:
        return """
        🗺️ **ESTADOS DE USA PARA BUSCAR PROPIEDADES:**

        📍 **Populares:**
        • California (CA) - Los Ángeles, San Francisco, Sacramento
        • Texas (TX) - Houston, Dallas, Austin
        • Florida (FL) - Miami, Orlando, Tampa
        • Nueva York (NY) - Nueva York, Buffalo, Rochester
        • Illinois (IL) - Chicago

        📍 **Otros estados:**
        • Washington (WA) - Seattle
        • Massachusetts (MA) - Boston
        • Colorado (CO) - Denver
        • Arizona (AZ) - Phoenix
        • Nevada (NV) - Las Vegas

        💡 Ejemplo: "buscar casas en Miami" o "propiedades en Texas"
        """

    # ===== AYUDA =====
    elif 'ayuda' in comando:
        return """
        📢 **COMANDOS DISPONIBLES:**
        ─────────────────────────────────────────────────────────────

        🏠 **PROPIEDADES EN USA:**
        • "buscar casas en Miami" - Busca propiedades en venta
        • "propiedades en Texas" - Busca en cualquier estado
        • "real estate Orlando" - Búsqueda de inmuebles
        • "estados para buscar" - Lista de estados disponibles

        ⏰ **TIEMPO:**
        • "hora" - Muestra la hora actual
        • "fecha" - Muestra la fecha actual

        🔍 **BÚSQUEDAS:**
        • "buscar google [algo]" - Busca en Google

        📝 **NOTAS:**
        • "agregar nota [texto]" - Guarda una nota
        • "mostrar notas" - Ver notas guardadas
        • "borrar notas" - Eliminar notas

        😊 **EXTRAS:**
        • "chiste" - Cuenta un chiste
        • "calcula 5 + 3" - Calculadora
        • "hola" - Saludo
        • "ayuda" - Este menú
        • "salir" - Cerrar chat

        ─────────────────────────────────────────────────────────────
        💡 Los enlaces abren Zillow directamente para ver propiedades reales
        """

    # ===== SALIR =====
    elif 'salir' in comando or 'adios' in comando:
        return "👋 ¡Hasta luego!"

    else:
        return f"❓ No entendí '{comando}'. Escribe 'ayuda' para ver comandos"


def formatear_propiedades(resultado, urls):
    """Formatea la respuesta de propiedades"""
    ubicacion = resultado['ubicacion'].upper()
    propiedades = resultado['propiedades']

    respuesta = f"""
    🏠 **PROPIEDADES EN {ubicacion}** (Datos referenciales)

    """

    for i, prop in enumerate(propiedades, 1):
        respuesta += f"""
    {i}. 📍 **{prop['direccion']}**
       💰 Precio: ${prop['precio']}
       🛏️ Habitaciones: {prop['habitaciones']} | 🚿 Baños: {prop['baños']}
       📐 Superficie: {prop['superficie']}

    """

    respuesta += f"""
    ─────────────────────────────────────────
    🔗 **VER PROPIEDADES REALES EN ZILLOW:**

    • [Ver en Zillow - General]({urls['url_principal']})
    • [Ver en Zillow - Solo venta]({urls['url_venta']})
    • [Ver en Zillow - Mapa]({urls['url_mapa']})
    ─────────────────────────────────────────
    """

    return respuesta


# ============================================
# INTERFAZ DE CHAT
# ============================================

input_box = widgets.Textarea(
    placeholder='Escribe tu comando aquí...\n\n📌 EJEMPLOS:\n• buscar casas en Miami\n• propiedades en Texas\n• real estate Orlando\n• buscar google inteligencia artificial\n• hora\n• chiste\n• ayuda',
    layout=widgets.Layout(width='100%', height='120px'),
    style={'description_width': 'initial'}
)

send_button = widgets.Button(
    description='📤 ENVIAR',
    button_style='primary',
    layout=widgets.Layout(width='auto')
)

clear_button = widgets.Button(
    description='🗑️ LIMPIAR CHAT',
    button_style='danger',
    layout=widgets.Layout(width='auto')
)

chat_output = widgets.Output()

def add_to_chat(message, is_user=True):
    with chat_output:
        if is_user:
            print(f"\n{'─'*55}")
            print(f"👤 TÚ: {message}")
        else:
            print(f"🤖 JARVIS: {message}")

def on_send_click(b):
    comando = input_box.value
    if comando.strip():
        add_to_chat(comando, is_user=True)

        respuesta = procesar_comando(comando)

        # Mostrar respuesta
        with chat_output:
            print(f"🤖 JARVIS: {respuesta}")

        input_box.value = ''

        if 'salir' in comando.lower():
            send_button.disabled = True
            print("\n💡 Vuelve a ejecutar la celda para reiniciar Jarvis")

def on_clear_click(b):
    with chat_output:
        clear_output(wait=True)
        print("✅ Chat limpiado")
        print("💬 Escribe 'ayuda' para ver comandos")
        print("🏠 Para buscar propiedades: 'buscar casas en Miami'")

send_button.on_click(on_send_click)
clear_button.on_click(on_clear_click)

display(widgets.VBox([
    widgets.HTML("<h3 style='text-align:center'>🤖 JARVIS - ASISTENTE CON BÚSQUEDA DE PROPIEDADES</h3>"),
    widgets.HTML("<p style='text-align:center; color:#666'>🏠 Busca casas en venta en cualquier ciudad de Estados Unidos</p>"),
    widgets.HBox([send_button, clear_button]),
    input_box,
    widgets.HTML("<b>💬 CHAT:</b>"),
    chat_output
]))

with chat_output:
    print("✅ Jarvis está listo")
    print("="*55)
    print("🏠 **COMANDOS DE EJEMPLO:**")
    print("   • 'buscar casas en Miami'")
    print("   • 'propiedades en Texas'")
    print("   • 'real estate Orlando'")
    print("   • 'estados para buscar' - Ver estados disponibles")
    print("="*55)
    print("💬 Escribe 'ayuda' para ver todos los comandos")

🤖 JARVIS - ASISTENTE CON BÚSQUEDA DE PROPIEDADES EN USA
✅ Busca casas en venta en cualquier ciudad de Estados Unidos
✅ Totalmente GRATIS - Sin necesidad de API key
